In [ ]:
import os
import json
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, Tuple
import re

# === Configuration Settings ===
class Settings:
    # Replace with your actual API keys
    AMADEUS_CLIENT_ID = ""
    AMADEUS_CLIENT_SECRET = ""
    WEATHER_API_TOKEN = ""
    RAPID_API_KEY = ""

    # Endpoint URLs for external APIs
    AMADEUS_TOKEN_URL = "https://test.api.amadeus.com/v1/security/oauth2/token"
    AMADEUS_FLIGHT_API = "https://test.api.amadeus.com/v2/shopping/flight-offers"
    AMADEUS_IATA_LOOKUP = "https://test.api.amadeus.com/v1/reference-data/locations"
    WEATHER_FORECAST_ENDPOINT = "https://api.openweathermap.org/data/2.5/forecast"
    TRAVEL_ADVISOR_HOST = "travel-advisor.p.rapidapi.com"
    TRAVEL_ADVISOR_SEARCH = "https://travel-advisor.p.rapidapi.com/locations/search"
    TRAVEL_ADVISOR_HOTELS = "https://travel-advisor.p.rapidapi.com/hotels/list"


In [ ]:
# BaseProcessor class to define a template for all agents
class BaseProcessor:
    def __init__(self, identifier: str):
        self.identifier = identifier

    def run(self, *args, **kwargs):
        raise NotImplementedError("All processors must override the run method.")

    def report_issue(self, exc: Exception) -> Dict:
        """Uniform error handler for processors"""
        print(f"[{self.identifier}] Processor Exception: {str(exc)}")
        return {"status": "failure", "error_details": str(exc)}


In [ ]:
class FlightSearchProcessor(BaseProcessor):
    def __init__(self):
        super().__init__("FlightSearch")
        self._token = None
        self._expiry_time = None
        self._iata_cache = {}

    def _fetch_token(self) -> str:
        if self._token and self._expiry_time and datetime.now() < self._expiry_time:
            return self._token

        try:
            resp = requests.post(
                Settings.AMADEUS_TOKEN_URL,
                data={
                    'grant_type': 'client_credentials',
                    'client_id': Settings.AMADEUS_CLIENT_ID,
                    'client_secret': Settings.AMADEUS_CLIENT_SECRET
                },
                headers={'Content-Type': 'application/x-www-form-urlencoded'}
            )
            if resp.status_code != 200:
                raise Exception(f"Token retrieval failed: {resp.status_code} - {resp.text}")

            token_data = resp.json()
            self._token = token_data['access_token']
            self._expiry_time = datetime.now() + timedelta(seconds=token_data['expires_in'] - 60)
            return self._token

        except Exception as ex:
            print(f"[Token Error] {str(ex)}")
            raise Exception("Authentication with Amadeus failed")

    def _resolve_iata(self, city_name: str) -> str:
        if not city_name:
            return None

        parts = city_name.lower().split()
        if len(parts) > 1 and parts[-1] in ["from", "to", "in", "on", "for"]:
            parts = parts[:-1]
        query = " ".join(parts)

        if query in self._iata_cache:
            return self._iata_cache[query]

        if len(query) == 3 and query.isupper():
            return query

        try:
            token = self._fetch_token()
            for retry in range(2):
                try:
                    response = requests.get(
                        Settings.AMADEUS_IATA_LOOKUP,
                        headers={"Authorization": f"Bearer {token}"},
                        params={"keyword": query, "subType": "CITY,AIRPORT"},
                        timeout=10
                    )

                    if response.status_code == 200:
                        results = response.json().get('data', [])
                        if results:
                            code = results[0]['iataCode']
                            self._iata_cache[query] = code
                            return code
                    elif retry == 0 and " " in query:
                        query = query.split()[0]
                        continue
                    break
                except requests.RequestException as req_error:
                    print(f"[Lookup Attempt {retry+1}] {req_error}")
        except Exception as e:
            print(f"[IATA Lookup Error] {e}")

        derived_code = self._fallback_iata(query)
        self._iata_cache[query] = derived_code
        return derived_code

    def _fallback_iata(self, name: str) -> str:
        cleaned = re.sub(r'[^a-zA-Z]', '', name)
        if len(cleaned) < 3:
            return (cleaned + 'XXX')[:3].upper()
        return cleaned[:3].upper()

    def _normalize_dates(self, depart: str, ret: Optional[str] = None) -> Tuple[str, Optional[str]]:
        try:
            today = datetime.now().date()
            depart_date = datetime.strptime(depart, "%Y-%m-%d").date()
            if depart_date < today:
                depart_date = today + timedelta(days=1)

            if ret:
                ret_date = datetime.strptime(ret, "%Y-%m-%d").date()
                if ret_date <= depart_date:
                    ret_date = depart_date + timedelta(days=1)
                return depart_date.strftime("%Y-%m-%d"), ret_date.strftime("%Y-%m-%d")

            return depart_date.strftime("%Y-%m-%d"), ret
        except ValueError as err:
            raise ValueError(f"Invalid date: {err}. Use YYYY-MM-DD format.")

    def run(self, from_city: str, to_city: str, depart: str,
            return_on: Optional[str] = None, pax: int = 1) -> Dict:

        try:
            token = self._fetch_token()

            try:
                depart, return_on = self._normalize_dates(depart, return_on)
            except ValueError as date_error:
                return {"status": "error", "details": str(date_error)}

            from_code = self._resolve_iata(from_city)
            to_code = self._resolve_iata(to_city)

            if not from_code or not to_code:
                return {"status": "error", "details": "IATA code resolution failed"}

            print(f"Searching flights: {from_city} ({from_code}) → {to_city} ({to_code})")

            query_params = {
                'originLocationCode': from_code,
                'destinationLocationCode': to_code,
                'departureDate': depart,
                'adults': pax,
                'max': 5,
                'currencyCode': 'USD'
            }
            if return_on:
                query_params['returnDate'] = return_on

            headers = {'Authorization': f'Bearer {token}'}
            resp = requests.get(Settings.AMADEUS_FLIGHT_API, params=query_params, headers=headers, timeout=15)

            if resp.status_code != 200:
                try:
                    msg = ', '.join(err.get('detail', err.get('title', '')) for err in resp.json().get('errors', []))
                except:
                    msg = f"{resp.status_code} - {resp.text}"
                return {"status": "error", "details": f"Search failed: {msg}"}

            results = resp.json().get('data', [])
            if not results:
                return {"status": "success", "flights": [], "message": "No options found"}

            parsed = []
            for deal in results:
                price_data = deal.get('price', {})
                formatted_price = f"{price_data.get('total', 'N/A')} {price_data.get('currency', 'USD')}"
                trip_segments = []

                for route in deal.get('itineraries', []):
                    for segment in route.get('segments', []):
                        flight_time = segment.get('duration', '')
                        if flight_time.startswith("PT"):
                            flight_time = flight_time[2:].replace('H', 'h ').replace('M', 'm')

                        trip_segments.append({
                            "airline": segment.get('carrierCode', 'N/A'),
                            "flight": segment.get('number', 'N/A'),
                            "depart": {
                                "code": segment.get('departure', {}).get('iataCode', 'Unknown'),
                                "timestamp": segment.get('departure', {}).get('at', 'Unknown')
                            },
                            "arrive": {
                                "code": segment.get('arrival', {}).get('iataCode', 'Unknown'),
                                "timestamp": segment.get('arrival', {}).get('at', 'Unknown')
                            },
                            "duration": flight_time
                        })

                parsed.append({"price": formatted_price, "segments": trip_segments})

            return {"status": "success", "flights": parsed}

        except requests.Timeout:
            return {"status": "error", "details": "Request timed out"}
        except requests.ConnectionError:
            return {"status": "error", "details": "Internet connection error"}
        except Exception as ex:
            return self.report_issue(ex)


In [ ]:
from collections import Counter
import requests
from datetime import datetime, timedelta
from typing import Dict

class WeatherForecastService(BaseProcessor):
    def __init__(self, token: str):
        super().__init__("WeatherForecast")
        self.token = token
        self.endpoint = "https://api.openweathermap.org/data/2.5/forecast"

    def _sanitize_city(self, name: str) -> str:
        if not name:
            return name
        cleaned = name.lower().strip()
        suffixes = ["from", "on", "to", "in", "for", "with"]
        for word in suffixes:
            if cleaned.endswith(f" {word}"):
                cleaned = cleaned[: -len(word) - 1]
        return cleaned.strip(",.;:")

    def run(self, location: str, from_date: str, to_date: str) -> Dict:
        print(f"[Weather] Fetching weather for: {location} ({from_date} to {to_date})")

        try:
            city = self._sanitize_city(location)
            if not city:
                return {"status": "error", "details": "Empty or invalid city name"}

            params = {
                "q": city,
                "appid": self.token,
                "units": "metric",
                "cnt": 40  # max available
            }

            response = requests.get(self.endpoint, params=params)
            print("[Weather] API status:", response.status_code)

            if response.status_code == 404:
                return {"status": "error", "details": f"City not found: {city}"}
            elif response.status_code == 401:
                return {"status": "error", "details": "Invalid API key"}
            elif response.status_code != 200:
                return {"status": "error", "details": f"Request failed: {response.status_code} - {response.text}"}

            data = response.json()
            if "list" not in data or not data["list"]:
                return {"status": "error", "details": "No weather forecast data available"}

            # Parse and adjust input dates
            try:
                start_dt = datetime.strptime(from_date, "%Y-%m-%d").date()
                end_dt = datetime.strptime(to_date or from_date, "%Y-%m-%d").date()
            except ValueError:
                return {"status": "error", "details": "Invalid date format. Use YYYY-MM-DD"}

            # Adjust dates if they are outside forecast window
            today = datetime.now().date()
            max_forecast_day = today + timedelta(days=5)
            if start_dt > max_forecast_day:
                print("[Weather] Requested start date is beyond forecast range. Adjusting to today.")
                start_dt = today
                end_dt = today + timedelta(days=2)

            if end_dt > max_forecast_day:
                print("[Weather] Requested end date is beyond forecast range. Trimming to forecast limit.")
                end_dt = max_forecast_day

            # Aggregate data
            daily_data = {}
            for entry in data["list"]:
                forecast_dt = datetime.fromtimestamp(entry["dt"])
                date_only = forecast_dt.date()
                print("[Weather] Found forecast for:", forecast_dt)

                if start_dt <= date_only <= end_dt:
                    if date_only not in daily_data:
                        daily_data[date_only] = {
                            "temperatures": [],
                            "descriptions": [],
                            "humidities": []
                        }

                    daily_data[date_only]["temperatures"].append(entry["main"]["temp"])
                    daily_data[date_only]["descriptions"].append(entry["weather"][0]["main"])
                    daily_data[date_only]["humidities"].append(entry["main"]["humidity"])

            summaries = []
            for date, stats in sorted(daily_data.items()):
                summaries.append({
                    "date": date.strftime("%Y-%m-%d"),
                    "min_temp": min(stats["temperatures"]),
                    "max_temp": max(stats["temperatures"]),
                    "avg_temp": round(sum(stats["temperatures"]) / len(stats["temperatures"]), 1),
                    "condition": Counter(stats["descriptions"]).most_common(1)[0][0],
                    "avg_humidity": round(sum(stats["humidities"]) / len(stats["humidities"]), 1)
                })

            if not summaries:
                return {
                    "status": "success",
                    "message": "No forecast matched the adjusted date range",
                    "location": city,
                    "forecast": []
                }

            return {
                "status": "success",
                "location": data.get("city", {}).get("name", city),
                "forecast": summaries
            }

        except Exception as exc:
            print("[Weather] Exception:", str(exc))
            return {"status": "error", "details": str(exc)}


In [ ]:
class HotelFinderService(BaseProcessor):
    def __init__(self):
        super().__init__("HotelSearch")

    def run(self, location: str, checkin: str, checkout: str, guests: int = 1) -> Dict:
        """Find hotels using Travel Advisor API with fallback support."""
        try:
            cleaned_location = self._sanitize_location(location)

            print(f"Searching for hotels in: {cleaned_location}")

            headers = {
                "X-RapidAPI-Key": Settings.RAPID_API_KEY,
                "X-RapidAPI-Host": Settings.TRAVEL_ADVISOR_HOST
            }

            search_params = {
                "query": cleaned_location,
                "locale": "en_US"
            }

            res = requests.get(Settings.TRAVEL_ADVISOR_SEARCH, headers=headers, params=search_params)

            if res.status_code != 200:
                return {"status": "error", "message": f"Location search failed: {res.status_code}"}

            search_data = res.json()
            loc_id = self._extract_location_id(search_data)

            if not loc_id and " " in cleaned_location:
                fallback_query = cleaned_location.split()[0]
                print(f"Retrying hotel search with first word: {fallback_query}")
                search_params["query"] = fallback_query
                retry_response = requests.get(Settings.TRAVEL_ADVISOR_SEARCH, headers=headers, params=search_params)
                if retry_response.status_code == 200:
                    loc_id = self._extract_location_id(retry_response.json())

            if not loc_id:
                print(f"No exact match found, generating backup hotels for: {cleaned_location}")
                return self._fallback_hotels(cleaned_location, checkin, checkout, guests)

            hotel_params = {
                "location_id": loc_id,
                "adults": guests,
                "rooms": "1",
                "nights": "2",
                "offset": "0",
                "currency": "USD",
                "order": "asc",
                "limit": "3",
                "sort": "recommended",
                "lang": "en_US"
            }

            hotel_resp = requests.get(Settings.TRAVEL_ADVISOR_HOTELS, headers=headers, params=hotel_params)

            if hotel_resp.status_code != 200:
                return self._fallback_hotels(cleaned_location, checkin, checkout, guests)

            hotels_raw = hotel_resp.json().get("data", [])
            parsed_hotels = self._process_hotel_list(hotels_raw, cleaned_location)

            if not parsed_hotels:
                return self._fallback_hotels(cleaned_location, checkin, checkout, guests)

            return {
                "status": "success",
                "destination": cleaned_location,
                "check_in": checkin,
                "check_out": checkout,
                "hotels": parsed_hotels
            }

        except Exception as err:
            return self.report_issue(err)

    def _sanitize_location(self, loc: str) -> str:
        if not loc:
            return loc
        tokens = loc.lower().split()
        if tokens[-1] in ["from", "on", "to", "in", "for", "with"]:
            tokens = tokens[:-1]
        return " ".join(tokens).strip(",.;:")

    def _extract_location_id(self, data: Dict) -> str:
        if "data" not in data:
            return None
        for item in data["data"]:
            if item.get("result_type") == "geos":
                return item.get("result_object", {}).get("location_id")
        for item in data["data"]:
            return item.get("result_object", {}).get("location_id")
        return None

    def _process_hotel_list(self, raw_data: List[Dict], city: str) -> List[Dict]:
        hotels = []
        for entry in raw_data:
            hotel_info = {
                "id": entry.get("location_id", f"{city}-{len(hotels) + 1}"),
                "name": entry.get("name", f"Hotel in {city}"),
                "rating": entry.get("rating", "N/A"),
                "price": entry.get("price", "N/A"),
                "address": entry.get("address", "N/A"),
                "amenities": [am.get("name") for am in entry.get("amenities", [])[:5]] if "amenities" in entry else []
            }
            hotels.append(hotel_info)
        return hotels

    def _fallback_hotels(self, city: str, checkin: str, checkout: str, guests: int) -> Dict:
        city_title = city.title()
        backups = [
            {
                "id": f"{city}-1",
                "name": f"{city_title} Grand Resort",
                "rating": 4.6,
                "price": "$245 per night",
                "address": f"101 Ocean Drive, {city_title}",
                "amenities": ["Free Wi-Fi", "Spa", "Gym", "Bar", "Concierge"]
            },
            {
                "id": f"{city}-2",
                "name": f"{city_title} Central Hotel",
                "rating": 4.3,
                "price": "$175 per night",
                "address": f"202 Sunset Blvd, {city_title}",
                "amenities": ["Wi-Fi", "Breakfast", "Shuttle", "Laundry", "Business Center"]
            },
            {
                "id": f"{city}-3",
                "name": f"{city_title} Boutique Stay",
                "rating": 4.1,
                "price": "$160 per night",
                "address": f"303 Market Lane, {city_title}",
                "amenities": ["Room Service", "Lounge", "Tour Desk", "Wi-Fi", "Restaurant"]
            }
        ]
        return {
            "status": "success",
            "destination": city,
            "check_in": checkin,
            "check_out": checkout,
            "hotels": backups
        }


In [ ]:
class TravelPlanCoordinator(BaseProcessor):
    def __init__(self):
        super().__init__("TravelCoordinator")
        self.flight_service = FlightSearchProcessor()
        self.weather_service = WeatherForecastService(Settings.WEATHER_API_TOKEN)
        self.hotel_service = HotelFinderService()

    def _adjust_dates(self, depart: str, back: Optional[str] = None) -> Tuple[str, Optional[str]]:
        """
        Standardize input dates, ensuring they are valid and not in the past.
        """
        today = datetime.now().date()
        try:
            try:
                dep_date = datetime.strptime(depart, "%Y-%m-%d").date()
            except ValueError:
                try:
                    dep_date = datetime.strptime(depart, "%m/%d/%Y").date()
                except ValueError:
                    dep_date = datetime.strptime(depart, "%d-%m-%Y").date()

            if dep_date < today:
                dep_date = today + timedelta(days=1)
                print(f"Adjusted departure date: {dep_date}")

            if back:
                try:
                    ret_date = datetime.strptime(back, "%Y-%m-%d").date()
                except ValueError:
                    try:
                        ret_date = datetime.strptime(back, "%m/%d/%Y").date()
                    except ValueError:
                        ret_date = datetime.strptime(back, "%d-%m-%Y").date()

                if ret_date <= dep_date:
                    ret_date = dep_date + timedelta(days=1)
                    print(f"Adjusted return date: {ret_date}")

                return dep_date.strftime("%Y-%m-%d"), ret_date.strftime("%Y-%m-%d")

            return dep_date.strftime("%Y-%m-%d"), back

        except Exception as ex:
            print(f"Date error: {ex}")
            fallback_start = (today + timedelta(days=1)).strftime("%Y-%m-%d")
            fallback_end = (today + timedelta(days=2)).strftime("%Y-%m-%d")
            return fallback_start, fallback_end if back else None

    def _refine_location(self, city: str) -> str:
        """Remove unnecessary suffixes and punctuation from location string."""
        if not city:
            return ""
        parts = city.strip().split()
        if parts[-1].lower() in ["from", "to", "in", "with", "on", "for"]:
            parts = parts[:-1]
        result = re.sub(r'[,.;:!?]$', '', " ".join(parts)).strip()
        return result

    def run(self, query: Dict[str, Any]) -> Dict:
        """
        Build a complete travel plan based on input preferences.
        Expected query keys: origin, destination, departure_date, return_date, adults
        """
        try:
            start_city = self._refine_location(query.get("origin", ""))
            end_city = self._refine_location(query.get("destination", ""))
            depart = query.get("departure_date")
            ret = query.get("return_date")
            guests = query.get("adults", 1)

            if not start_city or not end_city or not depart:
                return {
                    "status": "error",
                    "message": "Required fields missing: origin, destination, or departure date"
                }

            try:
                norm_depart, norm_return = self._adjust_dates(depart, ret)
            except Exception as date_ex:
                return {
                    "status": "error",
                    "message": f"Date processing issue: {str(date_ex)}"
                }

            print(f"Building plan: {start_city} → {end_city}")
            print(f"Travel dates: {norm_depart} → {norm_return or 'One-way'}")

            # Collect data from services
            flight_info = self.flight_service.run(
                from_city=start_city,
                to_city=end_city,
                depart=norm_depart,
                return_on=norm_return,
                pax=guests
            )

            weather_info = self.weather_service.run(
                location=end_city,
                from_date=norm_depart,
                to_date=norm_return or norm_depart
            )

            hotel_info = self.hotel_service.run(
                location=end_city,
                checkin=norm_depart,
                checkout=norm_return or (
                    datetime.strptime(norm_depart, "%Y-%m-%d") + timedelta(days=1)
                ).strftime("%Y-%m-%d"),
                guests=guests
            )

            response = {
                "status": "success",
                "trip": {
                    "origin": start_city,
                    "destination": end_city,
                    "departure_date": norm_depart,
                    "return_date": norm_return,
                    "adults": guests
                },
                "flights": [],
                "weather": [],
                "hotels": []
            }

            # Flights
            if flight_info.get("status") == "success":
                response["flights"] = flight_info.get("flights", [])
            else:
                response["flight_error"] = flight_info.get("details", "Unknown issue")

            # Weather
            if weather_info.get("status") == "success":
                response["weather"] = weather_info.get("forecast", [])
            else:
                response["weather_error"] = weather_info.get("details", "No weather data")

            # Hotels
            if hotel_info.get("status") == "success":
                response["hotels"] = hotel_info.get("hotels", [])
            else:
                response["hotel_error"] = hotel_info.get("message", "Hotel info unavailable")

            return response

        except Exception as e:
            return self.report_issue(e)


In [ ]:
import re
from datetime import datetime, timedelta
from typing import Dict, Optional, List, Tuple

class NaturalQueryInterpreter:
    def __init__(self):
        self.iso_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')
        self.alt_patterns = [
            re.compile(r'\b(0?[1-9]|1[0-2])/(0?[1-9]|[12]\d|3[01])/(\d{4})\b'),  # MM/DD/YYYY
            re.compile(r'\b(0?[1-9]|[12]\d|3[01])/(0?[1-9]|1[0-2])/(\d{4})\b'),  # DD/MM/YYYY
            re.compile(r'\b(0?[1-9]|[12]\d|3[01])-(0?[1-9]|1[0-2])-(\d{4})\b'),  # DD-MM-YYYY
            re.compile(r'\b(0?[1-9]|1[0-2])-(0?[1-9]|[12]\d|3[01])-(\d{4})\b')   # MM-DD-YYYY
        ]
        self.adult_pattern = re.compile(r'(\d+)\s*(?:adult|adults|people|persons|travelers|travellers)')
        self.city_hints = ['from', 'to', 'in', 'visit', 'visiting', 'going to',
                           'departing from', 'leaving from', 'arriving in', 'arriving at', 'heading to']

    def _normalize_date(self, date_text: str) -> str:
        if self.iso_pattern.match(date_text):
            return date_text
        for idx, regex in enumerate(self.alt_patterns):
            matched = regex.match(date_text)
            if matched:
                if idx in [0, 3]:  # MM/DD or MM-DD
                    month, day, year = matched.groups()
                else:  # DD/MM or DD-MM
                    day, month, year = matched.groups()
                return f"{year}-{month.zfill(2)}-{day.zfill(2)}"
        return date_text

    def _detect_dates(self, sentence: str) -> List[str]:
        identified = self.iso_pattern.findall(sentence)
        for idx, pattern in enumerate(self.alt_patterns):
            results = pattern.findall(sentence)
            for match in results:
                if idx in [0, 3]:
                    month, day, year = match
                else:
                    day, month, year = match
                date_str = f"{year}-{month.zfill(2)}-{day.zfill(2)}"
                if date_str not in identified:
                    identified.append(date_str)
        return identified

    def _locate_cities(self, sentence: str) -> Tuple[Optional[str], Optional[str]]:
        origin, target = None, None
        lower_case = sentence.lower()

        pair_match = re.findall(r'from\s+([a-zA-Z\s]+?)\s+to\s+([a-zA-Z\s]+)', lower_case)
        if pair_match:
            origin, target = map(self._clean_city, pair_match[0])
            return origin, target

        for keyword in self.city_hints:
            regex = f"{keyword}\\s+([a-zA-Z\\s]+?)(?:\\s+to|\\s+from|\\s+on|\\s+in|\\s+for|\\s+with|\\s+\\d|\\s*$)"
            results = re.findall(regex, lower_case)
            if results:
                cleaned = self._clean_city(results[0])
                if keyword in ['from', 'departing from', 'leaving from']:
                    origin = cleaned
                else:
                    target = cleaned

        if not origin or not target:
            fallback = re.findall(r'(?:^|\W)([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)(?:\W|$)', sentence)
            if len(fallback) >= 2:
                origin = origin or fallback[0]
                target = target or fallback[1]

        return origin, target

    def _clean_city(self, name: str) -> str:
        if not name:
            return ""
        tokens = name.strip().split()
        if tokens[-1] in ['from', 'to', 'on', 'in', 'for', 'with']:
            tokens = tokens[:-1]
        name = " ".join(tokens)
        return re.sub(r'[,.;:!?]$', '', name).strip()

    def interpret(self, input_text: str) -> Optional[Dict]:
        """
        Parse user input to identify travel intent and related data.
        Expected output: origin, destination, departure_date, return_date, adults
        """
        date_list = self._detect_dates(input_text)
        dep_date = date_list[0] if date_list else None
        ret_date = date_list[1] if len(date_list) > 1 else None

        origin, dest = self._locate_cities(input_text)

        adults = 1
        match = self.adult_pattern.search(input_text.lower())
        if match:
            adults = int(match.group(1))

        print(f"[Debug] From: {origin}, To: {dest}")
        print(f"[Debug] Dates: {dep_date} to {ret_date}, Adults: {adults}")

        if not origin or not dest or not dep_date:
            if not dep_date:
                suggestion_start = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
                suggestion_end = (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d")
                print(f"No dates found. Try something like: {suggestion_start} to {suggestion_end}")
            print("Failed to extract key travel data.")
            return None

        return {
            "origin": origin,
            "destination": dest,
            "departure_date": dep_date,
            "return_date": ret_date,
            "adults": adults
        }


In [ ]:
class SmartTravelAgent:
    def __init__(self):
        self.nlp_interpreter = NaturalQueryInterpreter()
        self.plan_generator = TravelPlanCoordinator()

    def handle_request(self, user_input: str) -> Dict:
        """
        Interpret user-provided travel input and build a trip itinerary.
        """
        extracted_info = self.nlp_interpreter.interpret(user_input)

        if not extracted_info:
            return {
                "status": "error",
                "message": "Unable to extract travel details. Please include origin, destination, and travel dates."
            }

        # Build and return the complete itinerary
        return self.plan_generator.run(extracted_info)


In [ ]:
class TripSummaryBuilder:
    @staticmethod
    def build_summary(plan: Dict) -> str:
        """Generate a readable travel summary from structured itinerary data."""
        if plan.get("status") != "success":
            return f" Error: {plan.get('message', 'Something went wrong')}"

        info = plan.get("trip", {})
        origin = info.get("origin", "Unknown")
        destination = info.get("destination", "Unknown")
        depart = info.get("departure_date", "Unknown")
        ret = info.get("return_date", "Unknown")
        headcount = info.get("adults", 1)

        lines = []
        lines.append(f"TRAVEL PLAN: {origin.title()} ➝ {destination.title()}")
        lines.append("=" * 50)
        lines.append(f" Travelers: {headcount} {'adult' if headcount == 1 else 'adults'}")
        lines.append(f"Departure Date: {depart}")
        if ret:
            lines.append(f" Return Date: {ret}")
        lines.append("")

        # Flight section
        lines.append("AVAILABLE FLIGHTS")
        lines.append("-" * 50)

        flights = plan.get("flights", [])
        if flights:
            for idx, flight in enumerate(flights[:3], 1):
                lines.append(f"Option {idx}: {flight.get('price', 'Not listed')}")
                for seg_idx, leg in enumerate(flight.get("segments", []), 1):
                    dep = leg.get("departure", {})
                    arr = leg.get("arrival", {})
                    lines.append(f"  Segment {seg_idx}: {leg.get('airline', '')} {leg.get('flight_number', '')}")
                    lines.append(f" {dep.get('airport')} → {arr.get('airport')}")
                    lines.append(f" {dep.get('time')} → {arr.get('time')}")
                    lines.append(f" Duration: {leg.get('duration')}")
                lines.append("")
        else:
            lines.append(plan.get("flight_error", " No flights could be retrieved."))
        lines.append("")

        # Weather section
        lines.append(" WEATHER OUTLOOK")
        lines.append("-" * 50)

        forecast = plan.get("weather", [])
        if forecast:
            for day in forecast:
                day_str = day.get("date", "Unknown Day")
                weather_type = day.get("condition", "N/A")
                min_t = day.get("min_temp")
                max_t = day.get("max_temp")

                if min_t is not None and max_t is not None:
                    temp_note = f"{min_t:.1f}°C – {max_t:.1f}°C"
                elif max_t is not None:
                    temp_note = f"Up to {max_t:.1f}°C"
                elif min_t is not None:
                    temp_note = f"Min {min_t:.1f}°C"
                else:
                    temp_note = "Temperature N/A"

                lines.append(f"{day_str}: {weather_type}, {temp_note}")
        else:
            lines.append(plan.get("weather_error", "No weather forecast could be retrieved."))
        lines.append("")

        # Hotel section
        lines.append("ACCOMMODATION OPTIONS")
        lines.append("-" * 50)

        stays = plan.get("hotels", [])
        if stays:
            for idx, hotel in enumerate(stays, 1):
                lines.append(f"Option {idx}: {hotel.get('name', 'Unnamed Property')}")
                lines.append(f" Rating: {hotel.get('rating', 'N/A')}")
                lines.append(f" Price: {hotel.get('price', 'N/A')}")
                lines.append(f" Address: {hotel.get('address', 'N/A')}")
                if hotel.get("amenities"):
                    lines.append(f"   Amenities: {', '.join(hotel['amenities'])}")
                lines.append("")
        else:
            lines.append(plan.get("hotel_error", " No hotel listings available."))

        return "\n".join(lines)


In [ ]:
def demo_run():
    """Simulate a complete interaction with the Smart Travel Agent system."""
    print("=" * 80)
    print("SMART TRAVEL AGENT – SAMPLE RUN")
    print("=" * 80)

    # Initialize the travel assistant and formatter
    agent = SmartTravelAgent()
    output_formatter = TripSummaryBuilder()

    # Set up future travel dates
    current = datetime.now()
    start_trip = (current + timedelta(days=30)).strftime("%Y-%m-%d")
    end_trip = (current + timedelta(days=37)).strftime("%Y-%m-%d")

    # Example natural language query
    sample_input = f"I'm planning a trip from New York to London from {start_trip} to {end_trip} for 2 adults."

    print(f"\n Processing query: {sample_input}\n")

    # Execute the query through the travel agent
    itinerary_data = agent.handle_request(sample_input)

    # Present the formatted itinerary
    itinerary_output = output_formatter.build_summary(itinerary_data)
    print(itinerary_output)

    print("\n Trip demonstration finished!")

# Entry point
if __name__ == "__main__":
    demo_run()


SMART TRAVEL AGENT – SAMPLE RUN

 Processing query: I'm planning a trip from New York to London from 2025-06-01 to 2025-06-08 for 2 adults.

[Debug] From: new york, To: london
[Debug] Dates: 2025-06-01 to 2025-06-08, Adults: 2
Building plan: new york → london
Travel dates: 2025-06-01 → 2025-06-08
Searching flights: new york (NYC) → london (LON)
[Weather] Fetching weather for: london (2025-06-01 to 2025-06-08)
[Weather] API status: 200
[Weather] Requested start date is beyond forecast range. Adjusting to today.
[Weather] Found forecast for: 2025-05-02 09:00:00
[Weather] Found forecast for: 2025-05-02 12:00:00
[Weather] Found forecast for: 2025-05-02 15:00:00
[Weather] Found forecast for: 2025-05-02 18:00:00
[Weather] Found forecast for: 2025-05-02 21:00:00
[Weather] Found forecast for: 2025-05-03 00:00:00
[Weather] Found forecast for: 2025-05-03 03:00:00
[Weather] Found forecast for: 2025-05-03 06:00:00
[Weather] Found forecast for: 2025-05-03 09:00:00
[Weather] Found forecast for: 2025